In [5]:
%pip install -q -U "langchain[google-genai]" langchain-anthropic langchain-openai langgraph python-dotenv pydantic pyyaml

Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
load_dotenv()  

import os
for k in ["GEMINI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    print(f"{k:20s}", "set" if os.environ.get(k) else "MISSING")

GEMINI_API_KEY       set
ANTHROPIC_API_KEY    set
OPENAI_API_KEY       set


In [7]:
OPENAI_MODEL = "gpt-4.1-nano"
GEMINI_MODEL = "gemini-3.1-flash-lite"
ANTHROPIC_MODEL = "claude-haiku-4.5"

In [17]:
from langchain.chat_models import init_chat_model

MODEL = f"google_genai:{GEMINI_MODEL}"   
model = init_chat_model(MODEL)

resp = model.invoke("Say hello to an AI engineering class in one sentence.")

print(resp.text) 


Welcome, class; today we begin our journey into the architecture and logic that define the future of artificial intelligence.


In [16]:
for model_id in ["google_genai:gemini-3.1-flash-lite", 
                 "anthropic:claude-3-5-haiku-latest",  
                 "openai:gpt-4.1-nano"]:  
    try:
        m = init_chat_model(model_id)
        print(model_id, "->", m.invoke("Reply with exactly one word: hello").text) 
    except Exception as e:
        print(model_id, "-> skipped:", type(e).__name__, "(install that integration / set its API key to try it)")


google_genai:gemini-3.1-flash-lite -> Hello
anthropic:claude-3-5-haiku-latest -> skipped: BadRequestError (install that integration / set its API key to try it)
openai:gpt-4.1-nano -> Hello


In [15]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.1-flash-lite"   
model = init_chat_model(MODEL)

resp = model.invoke("Say hello to an AI engineering class in one sentence.")

print(resp.text)


Welcome to the cutting edge of innovation—it’s time to start building the future of artificial intelligence.


In [14]:
for model_id in ["google_genai:gemini-3.1-flash-lite",
                 "anthropic:claude-3-5-haiku-latest", 
                 "openai:gpt-4o-mini"]:               
    try:
        m = init_chat_model(model_id)
        print(model_id, "->", m.invoke("Reply with exactly one word: hello").text)
    except Exception as e:
        print(model_id, "-> skipped:", type(e).__name__, "(install that integration / set its API key to try it)")


google_genai:gemini-3.1-flash-lite -> Hello
anthropic:claude-3-5-haiku-latest -> skipped: BadRequestError (install that integration / set its API key to try it)
openai:gpt-4o-mini -> Greetings!


In [12]:
from langchain.messages import SystemMessage, HumanMessage

SYSTEM = "You are a terse support assistant for Northstar Services. Answer in ONE sentence."
USER   = "A customer asks how to reset their password."

resp = model.invoke([SystemMessage(SYSTEM), HumanMessage(USER)])
print(resp.text)

Please visit the Northstar Services login page and click the "Forgot Password" link to receive reset instructions via email.


In [18]:
resp = model.invoke([
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": USER},
])
print(resp.text)

Please visit our website and click the "Forgot Password" link on the login page to receive a reset email.


In [19]:
from pydantic import BaseModel, Field
from typing import Literal

class Triage(BaseModel):
    category:  Literal["billing", "technical", "account", "general"] = Field(description="Main topic of the message")
    urgency:   Literal["low", "medium", "high"]                      = Field(description="How quickly this needs a response")
    sentiment: Literal["negative", "neutral", "positive"]            = Field(description="The customer's mood")
    summary:   str = Field(description="One-line summary of what the customer wants")

In [20]:
MESSAGE = "I've been charged twice for May and nobody has replied to my emails. This is ridiculous."

triage_model = model.with_structured_output(Triage)   
t = triage_model.invoke(MESSAGE)
print(type(t).__name__, "->", t)
print("category:", t.category, "| urgency:", t.urgency)  

Triage -> category='billing' urgency='high' sentiment='negative' summary='Customer reports a double charge for May and frustration over lack of email response.'
category: billing | urgency: high


In [21]:
triage_raw = model.with_structured_output(Triage, include_raw=True)
out = triage_raw.invoke(MESSAGE)

print("keys:", sorted(out.keys()))
print("parsed :", out["parsed"].category, out["parsed"].urgency)   
print("tokens :", out["raw"].usage_metadata)                       
print("error  :", out["parsing_error"])   

keys: ['parsed', 'parsing_error', 'raw']
parsed : billing high
tokens : {'input_tokens': 21, 'output_tokens': 42, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}}
error  : None


In [28]:
from langchain.agents import create_agent

def get_order_status(order_id: str) -> str:
    """Look up the delivery status of a customer order by its ID."""
    return f"Order {order_id}: shipped, arriving within 2 working days."

def refund_policy() -> str:
    """Return Northstar's refund policy in one sentence."""
    return "Refunds are available within 7 days of purchase for unused subscriptions."

agent = create_agent(
    model=MODEL,                                 
    tools=[get_order_status, refund_policy],
    system_prompt="You are a Northstar Services support assistant. Use tools when they help.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "Where is order A-2933, and what's your refund policy?"}
]})
print(result["messages"][-1].text)   

Order A-2933 has been shipped and is expected to arrive within 2 working days. 

Regarding our refund policy, refunds are available within 7 days of purchase for unused subscriptions.


In [29]:
for step in result["messages"]:
    step.pretty_print() 

================================ Human Message =================================

Where is order A-2933, and what's your refund policy?
================================== Ai Message ==================================

[]
Tool Calls:
  get_order_status (JBZC7kj3)
 Call ID: JBZC7kj3
  Args:
    order_id: A-2933
  refund_policy (oN5EZNcP)
 Call ID: oN5EZNcP
  Args:
================================= Tool Message =================================
Name: get_order_status

Order A-2933: shipped, arriving within 2 working days.
================================= Tool Message =================================
Name: refund_policy

Refunds are available within 7 days of purchase for unused subscriptions.
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Order A-2933 has been shipped and is expected to arrive within 2 working days. \n\nRegarding our refund policy, refunds are available within 7 days of purchase for unused subscriptions.', 'e

In [30]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(model=MODEL, tools=[], checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "customer-42"}}   
print(mem_agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ziaul."}]}, cfg)["messages"][-1].text)
print(mem_agent.invoke({"messages": [{"role": "user", "content": "What's my name?"}]},        cfg)["messages"][-1].text)

Hi Ziaul! It’s nice to meet you. How are you doing today? Is there anything I can help you with?
Your name is Ziaul!
